In [1]:
import numpy as np
import pandas as pd

import math
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from scipy.stats import chi2_contingency

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, auc, brier_score_loss
from sklearn.calibration import CalibrationDisplay
import torch
DEVICE = 'GPU' if torch.cuda.is_available() else 'CPU'

print(f"Using device: {DEVICE}")

Using device: GPU


In [2]:
df_external = pd.read_csv("external/f1_strategy_dataset_v4.csv")

In [3]:
df_external.head()

,Driver,LapNumber,Compound,Stint,TyreLife,Position,LapTime (s),Race,Year,LapTime_Delta,Cumulative_Degradation,PitStop,PitNextLap,RaceProgress,Normalized_TyreLife,Position_Change
0,ALB,1,MEDIUM,1,2.0,17,100.625,Abu Dhabi Grand Prix,2023,0.000,0.000,0,0,0.017241,0.117647,0.0
1,ALB,2,MEDIUM,1,3.0,18,93.560,Abu Dhabi Grand Prix,2023,-7.065,-7.065,0,0,0.034483,0.176471,-1.0
2,ALB,3,MEDIUM,1,4.0,18,91.768,Abu Dhabi Grand Prix,2023,-1.792,-8.857,0,0,0.051724,0.235294,0.0
3,ALB,4,MEDIUM,1,5.0,18,91.591,Abu Dhabi Grand Prix,2023,-0.177,-9.034,0,0,0.068966,0.294118,0.0
4,ALB,5,MEDIUM,1,6.0,18,91.422,Abu Dhabi Grand Prix,2023,-0.169,-9.203,0,0,0.086207,0.352941,0.0


In [4]:
df_external['Normalized_TyreLife_2'] = df_external['TyreLife'] / df_external.groupby(
    ['Year', 'Race', 'Driver', 'Stint']
)['TyreLife'].transform('max')

In [5]:
df_external.head()

,Driver,LapNumber,Compound,Stint,TyreLife,Position,LapTime (s),Race,Year,LapTime_Delta,Cumulative_Degradation,PitStop,PitNextLap,RaceProgress,Normalized_TyreLife,Position_Change,Normalized_TyreLife_2
0,ALB,1,MEDIUM,1,2.0,17,100.625,Abu Dhabi Grand Prix,2023,0.000,0.000,0,0,0.017241,0.117647,0.0,0.117647
1,ALB,2,MEDIUM,1,3.0,18,93.560,Abu Dhabi Grand Prix,2023,-7.065,-7.065,0,0,0.034483,0.176471,-1.0,0.176471
2,ALB,3,MEDIUM,1,4.0,18,91.768,Abu Dhabi Grand Prix,2023,-1.792,-8.857,0,0,0.051724,0.235294,0.0,0.235294
3,ALB,4,MEDIUM,1,5.0,18,91.591,Abu Dhabi Grand Prix,2023,-0.177,-9.034,0,0,0.068966,0.294118,0.0,0.294118
4,ALB,5,MEDIUM,1,6.0,18,91.422,Abu Dhabi Grand Prix,2023,-0.169,-9.203,0,0,0.086207,0.352941,0.0,0.352941


In [4]:
df_external[(df_external["Driver"]=="VER") & (df_external["PitNextLap"]==1) & (df_external["Year"]==2025)].head(20)

,Driver,LapNumber,Compound,Stint,TyreLife,Position,LapTime (s),Race,Year,LapTime_Delta,Cumulative_Degradation,PitStop,PitNextLap,RaceProgress,Normalized_TyreLife,Position_Change
54457,VER,34,INTERMEDIATE,4,34.0,3,128.598,Australian Grand Prix,2025,40.552,-15.011,0,1,0.435897,1.000000,0.0
54468,VER,46,MEDIUM,5,12.0,5,112.606,Australian Grand Prix,2025,19.732,-33.478,0,1,0.589744,0.375000,-4.0
56502,VER,40,HARD,1,40.0,1,108.754,Azerbaijan Grand Prix,2025,3.804,98.845,0,1,0.512821,1.000000,0.0
57572,VER,1,SOFT,1,4.0,8,103.877,Bahrain Grand Prix,2025,9.152,-1.010,0,1,0.012821,0.100000,-7.0
57573,VER,2,SOFT,1,5.0,8,98.403,Bahrain Grand Prix,2025,4.460,-28.361,0,1,0.025641,0.125000,-7.0
57574,VER,3,SOFT,1,6.0,8,98.475,Bahrain Grand Prix,2025,4.836,25.871,0,1,0.038462,0.150000,-7.0
57575,VER,4,SOFT,1,7.0,8,98.549,Bahrain Grand Prix,2025,4.805,81.740,0,1,0.051282,0.175000,-7.0
57576,VER,5,SOFT,1,8.0,7,98.760,Bahrain Grand Prix,2025,4.984,88.851,0,1,0.064103,0.200000,-6.0
57577,VER,6,SOFT,1,9.0,7,98.593,Bahrain Grand Prix,2025,4.947,88.684,0,1,0.076923,0.225000,-6.0
57578,VER,7,SOFT,1,10.0,7,98.929,Bahrain Grand Prix,2025,5.403,89.020,0,1,0.089744,0.250000,-6.0


In [5]:
df_train = pd.read_csv("data/train.csv")
print(df_train.columns)
print(df_external.columns)
print(set(df_train.columns).difference(set(df_external.columns)))
print(set(df_external.columns).difference(set(df_train.columns)))

Index(['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber',
       'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta',
       'Cumulative_Degradation', 'RaceProgress', 'Position_Change',
       'PitNextLap'],
      dtype='object')
Index(['Driver', 'LapNumber', 'Compound', 'Stint', 'TyreLife', 'Position',
       'LapTime (s)', 'Race', 'Year', 'LapTime_Delta',
       'Cumulative_Degradation', 'PitStop', 'PitNextLap', 'RaceProgress',
       'Normalized_TyreLife', 'Position_Change'],
      dtype='object')
{'id'}
{'Normalized_TyreLife'}


In [6]:
df_train=df_train.drop(['id'],axis=1)
df_external=df_external.drop(['Normalized_TyreLife'],axis=1)
df_train.head()

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [7]:
df_train=pd.concat([df_train, df_external], axis=0)
df_train=df_train.sample(frac=1)
df_train.reset_index(drop=True,inplace=True)
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 540511 entries, 0 to 540510
Data columns (total 15 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Driver                  540511 non-null  object 
 1   Compound                540445 non-null  object 
 2   Race                    540511 non-null  object 
 3   Year                    540511 non-null  int64  
 4   PitStop                 540511 non-null  int64  
 5   LapNumber               540511 non-null  int64  
 6   Stint                   540511 non-null  int64  
 7   TyreLife                540511 non-null  float64
 8   Position                540511 non-null  int64  
 9   LapTime (s)             540511 non-null  float64
 10  LapTime_Delta           540511 non-null  float64
 11  Cumulative_Degradation  540511 non-null  float64
 12  RaceProgress            540511 non-null  float64
 13  Position_Change         540511 non-null  float64
 14  PitNextLap          

In [9]:
df_train.describe()

,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
count,540511.000000,540511.000000,540511.000000,540511.000000,540511.000000,540511.000000,540511.000000,540511.000000,540511.000000,540511.000000,540511.000000,540511.000000
mean,2023.535948,0.157773,24.482301,1.837365,14.231582,9.654494,91.256021,-3.101221,-26.439742,0.355470,0.081628,0.209450
std,1.039448,0.364528,17.424550,0.955225,9.900556,5.303185,22.916342,44.233406,58.002225,0.256883,3.989509,0.406916
min,2022.000000,0.000000,1.000000,1.000000,1.000000,1.000000,67.012000,-2403.895000,-274.564000,0.012821,-18.000000,0.000000
25%,2023.000000,0.000000,10.000000,1.000000,6.000000,5.000000,82.430000,-8.717000,-47.611500,0.141026,-1.000000,0.000000
50%,2024.000000,0.000000,21.000000,2.000000,12.000000,10.000000,90.648000,-0.226000,-21.068000,0.294872,0.000000,0.000000
75%,2024.000000,0.000000,38.000000,2.000000,20.000000,14.000000,98.632000,0.338000,-5.918000,0.541667,2.000000,0.000000
max,2025.000000,1.000000,78.000000,8.000000,78.000000,20.000000,2526.253000,2433.472000,2412.431000,1.000000,18.000000,1.000000


In [10]:
df_train.isnull().sum()

Driver                     0
Compound                  66
Race                       0
Year                       0
PitStop                    0
LapNumber                  0
Stint                      0
TyreLife                   0
Position                   0
LapTime (s)                0
LapTime_Delta              0
Cumulative_Degradation     0
RaceProgress               0
Position_Change            0
PitNextLap                 0
dtype: int64

In [19]:
df_train.dropna(subset=["Compound"], inplace=True)

In [20]:
df_train['Driver'].value_counts()

Driver
NOR     6999
VER     6845
RUS     6707
HAM     6593
LEC     6587
        ... 
D705       1
D718       1
D689       1
D716       1
D710       1
Name: count, Length: 887, dtype: int64

In [21]:
df_train.duplicated().value_counts()

False    540445
Name: count, dtype: int64

In [22]:
df_train.isnull().sum()

Driver                    0
Compound                  0
Race                      0
Year                      0
PitStop                   0
LapNumber                 0
Stint                     0
TyreLife                  0
Position                  0
LapTime (s)               0
LapTime_Delta             0
Cumulative_Degradation    0
RaceProgress              0
Position_Change           0
PitNextLap                0
dtype: int64

In [25]:
df_train.to_csv("data/train_merged.csv",index=False)

In [24]:
len(df_train)

540445

In [ ]:
import pandas as pd
df_train=pd.read_csv('data/train.csv')
df_test=pd.read_csv('data/test.csv')
df_train_merged = pd.read_csv('data/train_merged.csv')